# Forest Memory — Gemma Ecological Reasoning

**Hackathon:** Kaggle Gemma 4 Good

This notebook loads `outputs/forest_memory_cases.json` and uses **Gemma** to generate
structured ecological resilience reports from multimodal environmental signals.

---

### What Gemma does here

Gemma is **not** a classifier, detector, or prediction engine.

Gemma acts as an **ecological reasoning engine**: it receives NDVI satellite data,
bioacoustic proxy scores, fire-history metadata, and uncertainty flags, then synthesises
a structured resilience report with careful proxy language.

---

> **Scientific disclaimer:** All acoustic and spectral-index values are proxy signals only.
> Reports do not represent confirmed species counts, validated biodiversity measurements,
> or wildfire predictions. Values are relative within this 4-site sample.

## 0  Setup

In [ ]:
import json
import os
import sys
import time
import textwrap
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path(".").resolve().parent
CASES_JSON  = ROOT / "outputs" / "forest_memory_cases.json"
OUT_DIR     = ROOT / "outputs" / "gemma_reports"
OUT_JSON    = OUT_DIR / "gemma_reports.json"
OUT_PRETTY  = OUT_DIR / "gemma_reports_pretty.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Cases JSON:", CASES_JSON)
print("Output dir:", OUT_DIR)

## 1  Configure Gemma API

Supports three authentication paths — whichever is available first is used:
1. **Kaggle Secrets** (`GOOGLE_API_KEY`) — preferred on Kaggle
2. **Environment variable** `GOOGLE_API_KEY`
3. **Manual entry** via `getpass`

In [ ]:
import google.generativeai as genai

# ── API key resolution ────────────────────────────────────────────────────────
api_key = None

# Path 1: Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    print("API key loaded from Kaggle Secrets.")
except Exception:
    pass

# Path 2: Environment variable
if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        print("API key loaded from environment variable.")

# Path 3: Manual entry
if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Google AI API key: ")

genai.configure(api_key=api_key)

# ── Model — MUST be Gemma 4 for this hackathon ────────────────────────────────
# Check available names: [m.name for m in genai.list_models()]
MODEL_NAME = "gemma-4-4b-it"       # E4B edge model — fast, Kaggle-friendly
# MODEL_NAME = "gemma-4-12b-it"    # richer reasoning if your quota allows
# MODEL_NAME = "gemma-4-27b-it"    # best quality for final submission

GENERATION_CONFIG = genai.types.GenerationConfig(
    temperature=0.3,         # low temperature → consistent scientific tone
    max_output_tokens=1200,
    top_p=0.9,
)

model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config=GENERATION_CONFIG,
    system_instruction=(
        "You are an ecological reasoning assistant for the Forest Memory project — "
        "a multimodal environmental intelligence system for conservation monitoring.\n\n"
        "You reason carefully from proxy signals derived from:\n"
        "- Satellite vegetation indices (NDVI, Sentinel-2)\n"
        "- Passive acoustic monitoring (bioacoustic proxies from WAV recordings)\n"
        "- Field metadata (fire history, land cover, invasive species pressure)\n\n"
        "STRICT SCIENTIFIC CONSTRAINTS — follow these at all times:\n"
        "1. Never claim exact species counts or confirmed species detections.\n"
        "2. Never predict wildfire occurrence or fire risk.\n"
        "3. Never diagnose ecosystem collapse or full recovery as definitive conclusions.\n"
        "4. Always use hedged language: 'proxy signal suggests', 'acoustic data may indicate', "
        "'satellite imagery is consistent with', 'uncertainty remains about'.\n"
        "5. Acknowledge data limitations and missing signals explicitly.\n"
        "6. Scores are relative within a 4-site sample — not absolute biodiversity indices.\n\n"
        "Your reports help conservation practitioners identify WHERE to look more closely, "
        "not WHAT the final ecological verdict is."
    ),
)

print(f"Model: {MODEL_NAME}  |  Temperature: {GENERATION_CONFIG.temperature}")
print("Gemma 4 ready.")

## 2  Load Forest Memory cases

In [ ]:
with open(CASES_JSON) as f:
    cases = json.load(f)

print(f"{len(cases)} cases loaded")
for c in cases:
    flags = [k for k, v in c["interpretation_flags"].items() if v]
    print(f"  {c['role']:30s}  site={c['site_id']}  flags={flags or ['none']}")

## 3  Prompt construction

In [ ]:
def format_case_for_prompt(case: dict) -> str:
    """Format a Forest Memory case dict into readable text for Gemma."""
    meta  = case.get("site_metadata", {})
    audio = case.get("audio", {})
    ndvi  = case.get("ndvi") or {}
    flags = case.get("interpretation_flags", {})

    def _s(v, fallback="not recorded"):
        return str(v) if v is not None else fallback

    def _score(col):
        d = audio.get(col, {})
        if not d or d.get("mean") is None:
            return "not available"
        return f"{d['mean']:.1f} / 100  (min {d['min']:.1f}, max {d['max']:.1f})"

    lines = [
        f"SITE ROLE:  {case['role'].replace('_', ' ').upper()}",
        f"SITE ID:    {case['site_id']}",
        "",
        "─── ECOSYSTEM CONTEXT ───────────────────────────────────────",
        f"  Land cover class :  {_s(meta.get('land_cover_class'))}",
        f"  Fire history class:  {_s(meta.get('fire_class'))}",
        f"  Vegetation age   :  {_s(meta.get('field_veld_age'))}",
        f"  Invasive pressure:  {_s(meta.get('field_aliens_within_20m'))}",
        f"  Elevation class  :  {_s(meta.get('elevation_class'))}",
        f"  Campaign season  :  {_s(meta.get('campaign'))}",
        "",
        "─── SATELLITE VEGETATION — NDVI (Sentinel-2, proxy) ─────────",
    ]

    if ndvi:
        lines += [
            f"  Mean NDVI        :  {ndvi['mean_ndvi']:.4f}  (0 = bare soil, 1 = dense green)",
            f"  NDVI variability :  std = {ndvi['std_ndvi']:.4f}  "
            f"(range {ndvi['min_ndvi']:.3f} – {ndvi['max_ndvi']:.3f})",
            f"  Images analysed  :  {ndvi['image_count']} (period {ndvi['period_start']} to {ndvi['period_end']})",
        ]
    else:
        lines.append("  NDVI data: not available for this site")

    lines += [
        "",
        "─── ACOUSTIC PROXIES (passive monitoring, 60 s WAV clips) ───",
        f"  WAV recordings           :  {audio.get('wav_file_count', '?')} files analysed",
        f"  Bioacoustic vitality     :  {_score('bioacoustic_vitality_score')}",
        f"  Acoustic richness        :  {_score('acoustic_richness_score')}",
        f"  Bird activity (proxy)    :  {_score('bird_activity_proxy')}",
        f"  Insect activity (proxy)  :  {_score('insect_activity_proxy_score')}",
        f"  Human noise interference :  {_score('human_disturbance_proxy')}",
        "",
        "─── INTERPRETATION FLAGS ────────────────────────────────────",
        f"  green_not_alive_signal       :  {flags.get('green_not_alive_signal', False)}",
        f"    (NDVI high but vitality low — possible mismatch between greenness and acoustic life)",
        f"  spatially_variable_vegetation:  {flags.get('spatially_variable_vegetation', False)}",
        f"    (high NDVI temporal std — patchy or heterogeneous vegetation)",
        f"  recent_fire_recovery_context :  {flags.get('recent_fire_recovery_context', False)}",
        f"    (fire history metadata indicates 1-to-6 year post-fire window)",
        f"  acoustic_uncertainty         :  {flags.get('acoustic_uncertainty', False)}",
        f"    (high low-frequency energy — wind/noise may mask biological signals)",
    ]

    return "\n".join(lines)


SITE_PROMPT_TEMPLATE = """\
Analyse the following ecosystem site data and produce an ecological resilience report.

{site_data}

Write a structured report with exactly these five sections.
Keep each section to 2–4 sentences.
Use careful proxy language throughout — never overclaim.

## Vegetation Interpretation
(What does the NDVI signal suggest about vegetation state? Note any tension between
 greenness and other signals, or flag missing data.)

## Bioacoustic Interpretation
(What do the acoustic proxy scores suggest about soundscape activity?
 Note dominant signals and any high uncertainty.)

## Multimodal Tension Summary
(Where do satellite and acoustic signals agree or diverge? What might this divergence —
 or alignment — indicate about the ecosystem state?)

## Recovery Interpretation
(Given fire history, land cover, and combined proxy data, what recovery trajectory
 does this site appear to be on? Be explicit about what is unknown.)

## Uncertainty Notes
(What data limitations, missing signals, seasonal confounders, or methodological
 caveats should a practitioner consider when interpreting this report?)
"""

# Quick preview of one formatted case
print(format_case_for_prompt(cases[0]))

## 4  Generate per-site reports

In [ ]:
def parse_sections(text: str) -> dict[str, str]:
    """Split Gemma's markdown response into the five named sections."""
    section_keys = [
        "Vegetation Interpretation",
        "Bioacoustic Interpretation",
        "Multimodal Tension Summary",
        "Recovery Interpretation",
        "Uncertainty Notes",
    ]
    sections: dict[str, str] = {}
    current_key = None
    buffer: list[str] = []

    for line in text.splitlines():
        stripped = line.strip()
        matched = next(
            (k for k in section_keys if stripped.startswith(f"## {k}")),
            None,
        )
        if matched:
            if current_key and buffer:
                sections[current_key] = " ".join(" ".join(buffer).split())
            current_key = matched
            buffer = []
        elif current_key and stripped and not stripped.startswith("#"):
            buffer.append(stripped)

    if current_key and buffer:
        sections[current_key] = " ".join(" ".join(buffer).split())

    # Fill any missing sections
    for k in section_keys:
        sections.setdefault(k, "(not generated)")

    return sections


def generate_site_report(case: dict, retry: int = 2) -> dict:
    """Call Gemma for one site. Returns a report dict."""
    prompt = SITE_PROMPT_TEMPLATE.format(
        site_data=format_case_for_prompt(case)
    )
    for attempt in range(retry + 1):
        try:
            response = model.generate_content(prompt)
            raw_text = response.text
            break
        except Exception as e:
            if attempt < retry:
                print(f"  Retry {attempt + 1}/{retry} after error: {e}")
                time.sleep(5)
            else:
                raw_text = f"[ERROR: {e}]"

    return {
        "role":     case["role"],
        "site_id":  case["site_id"],
        "sections": parse_sections(raw_text),
        "raw_response": raw_text,
        "model": MODEL_NAME,
        "proxy_disclaimer": case.get("proxy_disclaimer", ""),
    }


print("Generating reports — one API call per site …")
reports = []
for i, case in enumerate(cases):
    print(f"  [{i+1}/{len(cases)}] {case['role']} …", end=" ", flush=True)
    t0 = time.time()
    report = generate_site_report(case)
    elapsed = time.time() - t0
    reports.append(report)
    print(f"done ({elapsed:.1f}s)")
    if i < len(cases) - 1:
        time.sleep(2)   # polite rate-limiting

print(f"\n{len(reports)} reports generated.")

## 5  Save reports

In [ ]:
with open(OUT_JSON, "w") as f:
    json.dump(reports, f, separators=(",", ":"))

with open(OUT_PRETTY, "w") as f:
    json.dump(reports, f, indent=2)

print(f"Saved → {OUT_JSON.relative_to(ROOT)}")
print(f"Saved → {OUT_PRETTY.relative_to(ROOT)}")

## 6  Display reports

In [ ]:
ROLE_EMOJI = {
    "healthy_baseline":       "🌿",
    "burned_recovering":      "🔥",
    "invasive_disturbed":     "⚠️",
    "wet_dry_pair_complement": "💧",
}

SECTION_ORDER = [
    "Vegetation Interpretation",
    "Bioacoustic Interpretation",
    "Multimodal Tension Summary",
    "Recovery Interpretation",
    "Uncertainty Notes",
]

for report in reports:
    emoji = ROLE_EMOJI.get(report["role"], "📍")
    role_display = report["role"].replace("_", " ").title()

    md_parts = [
        f"---",
        f"## {emoji} {role_display}  ",
        f"**Site ID:** `{report['site_id']}`  |  **Model:** `{report['model']}`",
        "",
    ]
    for section in SECTION_ORDER:
        text = report["sections"].get(section, "(not generated)")
        md_parts.append(f"### {section}")
        md_parts.append(text)
        md_parts.append("")

    md_parts.append(
        f"> *{report['proxy_disclaimer']}*"
    )
    display(Markdown("\n".join(md_parts)))

## 7  Cross-site synthesis

A single prompt feeding all four sites to Gemma, asking for a comparative resilience narrative.

In [ ]:
# Build a condensed cross-site summary for the synthesis prompt
def _condensed(case: dict) -> str:
    meta  = case["site_metadata"]
    audio = case["audio"]
    ndvi  = case.get("ndvi") or {}
    flags = [k for k, v in case["interpretation_flags"].items() if v]

    vitality = (audio.get("bioacoustic_vitality_score") or {}).get("mean")
    ndvi_m   = ndvi.get("mean_ndvi")
    ndvi_s   = ndvi.get("std_ndvi")

    return (
        f"[{case['role']}]  "
        f"Land cover: {meta.get('land_cover_class', '?')}  "
        f"Fire: {meta.get('fire_class', '?')}  "
        f"Vitality proxy: {f'{vitality:.1f}/100' if vitality is not None else 'N/A'}  "
        f"NDVI mean: {f'{ndvi_m:.3f}' if ndvi_m else 'N/A'}  "
        f"NDVI std: {f'{ndvi_s:.3f}' if ndvi_s else 'N/A'}  "
        f"Flags: {', '.join(flags) if flags else 'none'}"
    )

cross_site_data = "\n".join(_condensed(c) for c in cases)

synthesis_prompt = f"""\
You have analysed four ecosystem monitoring sites from the BioSCape campaign
in the Cape Floristic Region, South Africa. Here is a summary of all four:

{cross_site_data}

Write a cross-site ecological resilience synthesis with these three sections.
2–4 sentences per section. Use careful proxy language throughout.

## Comparative Resilience Narrative
(Compare the four sites along the vitality–greenness axis. Which site shows the
 strongest mismatch between vegetation greenness and acoustic vitality? What might
 this suggest about the limits of using greenness alone as a resilience indicator?)

## Key Multimodal Insight
(What is the single most important observation that emerges from combining satellite
 and acoustic signals across these sites? Explain why multimodal data adds value
 that neither source alone provides.)

## Conservation Monitoring Implications
(What specific follow-up actions or observations would these proxy signals
 motivate for a field ecologist? Be concrete and uncertainty-aware.)
"""

print("Generating cross-site synthesis …")
t0 = time.time()
synthesis_response = model.generate_content(synthesis_prompt)
synthesis_text = synthesis_response.text
print(f"Done ({time.time() - t0:.1f}s)")

# Save alongside reports
synthesis_record = {
    "type": "cross_site_synthesis",
    "model": MODEL_NAME,
    "raw_response": synthesis_text,
    "proxy_disclaimer": (
        "All acoustic and NDVI values are proxy signals only. "
        "Do not interpret as confirmed species counts, validated biodiversity "
        "measurements, or wildfire predictions."
    ),
}

all_outputs = {"site_reports": reports, "synthesis": synthesis_record}
with open(OUT_PRETTY, "w") as f:
    json.dump(all_outputs, f, indent=2)
print(f"Updated → {OUT_PRETTY.relative_to(ROOT)}")

In [ ]:
SYNTHESIS_SECTIONS = [
    "Comparative Resilience Narrative",
    "Key Multimodal Insight",
    "Conservation Monitoring Implications",
]

synthesis_sections = parse_sections(synthesis_text)
# parse_sections looks for "## X" — map the synthesis-specific keys

md_parts = [
    "---",
    "## 🌍 Cross-Site Ecological Synthesis",
    f"**Model:** `{MODEL_NAME}`",
    "",
]
for section in SYNTHESIS_SECTIONS:
    text = synthesis_sections.get(section, "")
    if not text:
        # fallback: print raw if sections didn't parse cleanly
        md_parts += ["### Raw synthesis output", synthesis_text]
        break
    md_parts.append(f"### {section}")
    md_parts.append(text)
    md_parts.append("")

md_parts.append(
    "> *All values are acoustic and spectral-index proxies — "
    "not confirmed biodiversity measurements.*"
)
display(Markdown("\n".join(md_parts)))

## 8  Quick reference — report quality check

In [ ]:
# Flag any reports that are suspiciously short (possible API error or truncation)
MIN_CHARS = 150
print("Report quality check:")
print(f"{'Role':32s}  {'Sections OK':12s}  {'Min section len':16s}  Status")
print("-" * 80)
for r in reports:
    secs = r["sections"]
    n_ok = sum(1 for v in secs.values() if len(v) >= MIN_CHARS)
    min_len = min(len(v) for v in secs.values())
    status = "✅" if n_ok == 5 else f"⚠️  only {n_ok}/5 sections ≥{MIN_CHARS} chars"
    print(f"{r['role']:32s}  {n_ok}/5         {min_len:6d} chars     {status}")

---

## Output summary

| File | Contents |
|---|---|
| `outputs/gemma_reports/gemma_reports.json` | Compact JSON: all site reports + synthesis |
| `outputs/gemma_reports/gemma_reports_pretty.json` | Human-readable version |

---

**Next steps in the Forest Memory pipeline:**
- `04_litert_edge.ipynb` — LiteRT model quantisation for edge deployment
- `05_gradio_demo.py` — Interactive Gradio UI wiring reports → display
- Demo video production